# 05 — Synthetic Multi-Device Unification Demo

> **This notebook uses SYNTHETIC data** generated from public dataset distributions
> (LifeSnaps Fitbit + published WHOOP/Oura/Garmin/Strava norms).
> It demonstrates cross-device conflict resolution, provenance tracking,
> confidence scoring, and normalization — NOT real patient data.

**What this demonstrates:**
1. Raw schema differences across 5 providers (units, naming, nesting)
2. Normalization into a unified schema
3. Cross-device conflict resolution (same user, same metric, different values)
4. Provenance tracking (which device produced which reading)
5. Missingness patterns across devices and users

In [ ]:
import json
from pathlib import Path

import polars as pl
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="Set2")

RAW_DIR = Path("../data/raw")
NORM_DIR = Path("../data/normalized")

assert RAW_DIR.exists(), "Run scripts/seed_synthetic.py first"
assert NORM_DIR.exists(), "Run scripts/seed_synthetic.py first"

manifest = json.loads((NORM_DIR / "manifest.json").read_text())
print(f"Data type: {manifest['data_type']}")
print(f"Users: {manifest['n_users']}, Days: {manifest['n_days']}")
print(f"Date range: {manifest['date_range']}")
print(f"Raw files: {manifest['raw_files_per_provider']}")

## 1. Raw Schema Differences

Each provider uses different JSON structures, field names, units, and nesting.
This is the core challenge of wearable data unification.

In [ ]:
# Load one sample file from each provider
samples = {}
for provider in ["fitbit", "oura", "whoop", "garmin", "strava"]:
    files = sorted((RAW_DIR / provider).glob("*.json"))
    if files:
        samples[provider] = json.loads(files[0].read_text())

# Show sleep duration in each provider's native format
print("SLEEP DURATION — same concept, different representations:")
print("=" * 60)

if "fitbit" in samples:
    fb_dur = samples["fitbit"]["sleep"]["duration"]
    print(f"  Fitbit:  {fb_dur:>12,} ms  (duration field in milliseconds)")

if "oura" in samples:
    oura_dur = samples["oura"]["daily_sleep"]["total_sleep_duration"]
    print(f"  Oura:    {oura_dur:>12,} sec (total_sleep_duration in seconds)")

if "whoop" in samples:
    ws = samples["whoop"]["sleep"]["score"]["stage_summary"]
    whoop_dur = ws["total_light_sleep_time_milli"] + ws["total_slow_wave_sleep_time_milli"] + ws["total_rem_sleep_time_milli"]
    print(f"  WHOOP:   {whoop_dur:>12,} ms  (sum of stage_summary.*_milli fields)")

if "garmin" in samples:
    garmin_dur = samples["garmin"]["sleep"]["durationInSeconds"]
    print(f"  Garmin:  {garmin_dur:>12,} sec (durationInSeconds field)")

print()
print("After normalization: all become 'duration_minutes' (float)")

In [ ]:
# Show resting HR field paths
print("RESTING HEART RATE — different field paths:")
print("=" * 60)

if "fitbit" in samples:
    rhr = samples["fitbit"]["activity_summary"]["summary"]["restingHeartRate"]
    print(f"  Fitbit:  activity_summary.summary.restingHeartRate = {rhr}")

if "oura" in samples:
    rhr = samples["oura"]["daily_sleep"]["lowest_heart_rate"]
    print(f"  Oura:    daily_sleep.lowest_heart_rate = {rhr}")

if "whoop" in samples:
    rhr = samples["whoop"]["recovery"]["score"]["resting_heart_rate"]
    print(f"  WHOOP:   recovery.score.resting_heart_rate = {rhr}")

if "garmin" in samples:
    rhr = samples["garmin"]["daily_summary"]["restingHeartRateInBeatsPerMinute"]
    print(f"  Garmin:  daily_summary.restingHeartRateInBeatsPerMinute = {rhr}")

print()
print("After normalization: all become 'resting_hr' (float, bpm)")

## 2. Unified Normalized Tables

In [ ]:
daily = pl.read_parquet(NORM_DIR / "unified_daily_metrics.parquet")
sleep = pl.read_parquet(NORM_DIR / "unified_sleep.parquet")
hr = pl.read_parquet(NORM_DIR / "unified_heart_rate.parquet")
activities = pl.read_parquet(NORM_DIR / "unified_activities.parquet")

print(f"Daily metrics: {daily.shape} — sources: {daily['source'].unique().to_list()}")
print(f"Sleep:         {sleep.shape} — sources: {sleep['source'].unique().to_list()}")
print(f"Heart rate:    {hr.shape} — sources: {hr['source'].unique().to_list()}")
print(f"Activities:    {activities.shape} — sources: {activities['source'].unique().to_list()}")

print("\nDaily metrics sample:")
print(daily.head(5))

## 3. Cross-Device Conflict Resolution

When multiple devices report the same metric for the same user on the same day,
we need a strategy to resolve conflicts. This section demonstrates the problem
and a confidence-weighted approach.

In [ ]:
# Find dates where multiple providers report resting HR for the same user
hr_conflicts = (
    hr.filter(pl.col("resting_hr").is_not_null())
    .group_by(["user_id", "date"])
    .agg([
        pl.len().alias("n_sources"),
        pl.col("source").alias("sources"),
        pl.col("resting_hr").alias("values"),
        pl.col("confidence").alias("confidences"),
    ])
    .filter(pl.col("n_sources") > 1)
    .sort(["user_id", "date"])
)

print(f"HR conflicts: {len(hr_conflicts)} user-date pairs with multiple sources")
print()
print("Example conflicts (same user, same day, different HR readings):")
print(hr_conflicts.head(8))

In [ ]:
# Confidence-weighted merge: resolve conflicts by weighting each source
def confidence_weighted_merge(group_df: pl.DataFrame) -> pl.DataFrame:
    """Merge conflicting readings using confidence-weighted average."""
    merged_rows = []
    for (user_id, date), group in group_df.filter(
        pl.col("resting_hr").is_not_null()
    ).group_by(["user_id", "date"]):
        values = group["resting_hr"].to_list()
        confidences = group["confidence"].to_list()
        sources = group["source"].to_list()
        
        # Weighted average
        total_conf = sum(confidences)
        if total_conf > 0:
            merged_val = sum(v * c for v, c in zip(values, confidences)) / total_conf
        else:
            merged_val = np.mean(values)
        
        merged_rows.append({
            "user_id": user_id,
            "date": date,
            "resting_hr_merged": round(merged_val, 1),
            "n_sources": len(values),
            "sources": sources,
            "raw_values": values,
            "spread_bpm": round(max(values) - min(values), 1),
        })
    
    return pl.DataFrame(merged_rows)

merged_hr = confidence_weighted_merge(hr)
print(f"Merged HR readings: {len(merged_hr)} user-date pairs")
print(f"Average cross-device spread: {merged_hr['spread_bpm'].mean():.1f} bpm")
print(f"Max spread observed: {merged_hr['spread_bpm'].max():.1f} bpm")
print()
print(merged_hr.filter(pl.col("n_sources") > 1).head(10))

In [ ]:
# Visualize HR disagreement across devices
multi_source = merged_hr.filter(pl.col("n_sources") > 1)

if len(multi_source) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    # Distribution of spread
    spreads = multi_source["spread_bpm"].to_list()
    axes[0].hist(spreads, bins=15, edgecolor="black", alpha=0.7)
    axes[0].axvline(np.mean(spreads), color="red", linestyle="--", label=f"Mean: {np.mean(spreads):.1f} bpm")
    axes[0].set_xlabel("Cross-device HR Spread (bpm)")
    axes[0].set_ylabel("Count")
    axes[0].set_title("How Much Do Devices Disagree on Resting HR?")
    axes[0].legend()
    
    # Per-provider resting HR distributions
    hr_valid = hr.filter(pl.col("resting_hr").is_not_null()).to_pandas()
    sns.boxplot(data=hr_valid, x="source", y="resting_hr", ax=axes[1])
    axes[1].set_xlabel("Provider")
    axes[1].set_ylabel("Resting HR (bpm)")
    axes[1].set_title("Resting HR Distribution by Device")
    
    plt.tight_layout()
    plt.show()
else:
    print("No multi-source conflicts found")

## 4. Provenance Tracking

Every unified row carries its `source` field, enabling full traceability
back to the original device and raw payload.

In [ ]:
# Show provenance: for user_05 (power user), show all sources for a single day
user_05_id = "a1b2c3d4-5555-4000-8000-000000000005"
sample_date = "2024-03-05"

print(f"Provenance for user_05 on {sample_date}:")
print("=" * 60)

print("\nDaily metrics:")
u5_daily = daily.filter((pl.col("user_id") == user_05_id) & (pl.col("date") == sample_date))
if len(u5_daily) > 0:
    for row in u5_daily.iter_rows(named=True):
        print(f"  [{row['source']}] steps={row['steps']}, cals={row['calories']}, confidence={row['confidence']}")

print("\nSleep:")
u5_sleep = sleep.filter((pl.col("user_id") == user_05_id) & (pl.col("sleep_date") == sample_date))
if len(u5_sleep) > 0:
    for row in u5_sleep.iter_rows(named=True):
        print(f"  [{row['source']}] duration={row['duration_minutes']:.0f}min, efficiency={row['efficiency']}, confidence={row['confidence']}")

print("\nHeart rate:")
u5_hr = hr.filter((pl.col("user_id") == user_05_id) & (pl.col("date") == sample_date))
if len(u5_hr) > 0:
    for row in u5_hr.iter_rows(named=True):
        print(f"  [{row['source']}] resting_hr={row['resting_hr']}, confidence={row['confidence']}")

## 5. Missingness Heatmap

Visualize which users have data from which providers on which days.
This exposes the real-world challenge of incomplete coverage.

In [ ]:
# Build presence matrix: user x date x provider
users_info = {u["id"]: u["name"] for u in manifest["users"]}
all_dates = sorted(set(daily["date"].to_list() + sleep["sleep_date"].to_list() + hr["date"].to_list()))
providers = ["fitbit", "oura", "whoop", "garmin", "strava"]

# For each user, count days with data per provider
presence_data = []
for uid, uname in users_info.items():
    for provider in providers:
        # Count files in raw dir for this user+provider
        raw_files = list((RAW_DIR / provider).glob(f"{uid}_*.json"))
        presence_data.append({
            "user": uname,
            "provider": provider,
            "days_with_data": len(raw_files),
        })

presence_df = pl.DataFrame(presence_data)

# Pivot for heatmap
pivot = presence_df.pivot(on="provider", index="user", values="days_with_data").fill_null(0)
pivot_np = pivot.select(providers).to_numpy()
user_labels = pivot["user"].to_list()

fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(
    pivot_np,
    xticklabels=[p.title() for p in providers],
    yticklabels=user_labels,
    annot=True, fmt="d", cmap="YlGn",
    vmin=0, vmax=14,
    ax=ax,
)
ax.set_title(f"Data Availability: Days with Data per Provider (out of {manifest['n_days']})")
ax.set_ylabel("User")
plt.tight_layout()
plt.show()

In [ ]:
# Detailed day-by-day presence for user_05 (power user with all devices)
user_05_files = {}
for provider in providers:
    files = sorted((RAW_DIR / provider).glob(f"{user_05_id}_*.json"))
    dates_present = [f.stem.split("_")[-1] for f in files]
    user_05_files[provider] = set(dates_present)

# Build day-level grid
from datetime import date, timedelta
start = date(2024, 3, 1)
date_list = [(start + timedelta(days=i)).isoformat() for i in range(14)]

grid = np.zeros((len(providers), len(date_list)))
for i, prov in enumerate(providers):
    for j, d in enumerate(date_list):
        if d in user_05_files.get(prov, set()):
            grid[i, j] = 1

fig, ax = plt.subplots(figsize=(14, 3.5))
sns.heatmap(
    grid,
    xticklabels=[d[5:] for d in date_list],
    yticklabels=[p.title() for p in providers],
    cmap=["#f0f0f0", "#2ecc71"],
    cbar=False, linewidths=0.5,
    ax=ax,
)
ax.set_title("user_05 (Power User) — Day-by-Day Device Coverage")
ax.set_xlabel("Date")
plt.tight_layout()
plt.show()

# Count days with full coverage vs partial
full_days = sum(1 for j in range(14) if all(grid[i, j] for i in range(len(providers))))
print(f"\nuser_05: {full_days}/14 days with ALL 5 devices reporting")

## 6. Activity Double-Counting

When a user records a workout, it may appear in Strava, Garmin, AND WHOOP.
The unification layer needs to detect and deduplicate these.

In [ ]:
# Show activities for same user on same day from multiple sources
activity_dupes = (
    activities.group_by(["user_id", "date"])
    .agg([
        pl.len().alias("n_sources"),
        pl.col("source").alias("sources"),
        pl.col("activity_type").alias("types"),
        pl.col("distance_meters").alias("distances"),
        pl.col("calories").alias("calories_list"),
    ])
    .filter(pl.col("n_sources") > 1)
    .sort(["user_id", "date"])
)

print(f"Activity double/triple counting: {len(activity_dupes)} user-date pairs")
print()
if len(activity_dupes) > 0:
    print(activity_dupes.head(10))
    print()
    print("These represent the SAME workout recorded by multiple devices.")
    print("A deduplication strategy is needed (e.g., prefer GPS source, merge metadata).")

## Summary

This demo illustrates the core challenges that the health data unification
backend solves:

1. **Schema normalization** — Different JSON shapes, field names, and units
2. **Conflict resolution** — Multiple devices disagree on the same metric
3. **Provenance** — Every row traces back to its source device
4. **Confidence scoring** — Direct measurements vs derived/estimated values
5. **Missingness** — Incomplete coverage is the norm, not the exception
6. **Deduplication** — Same activity counted multiple times across devices

In [ ]:
print("=" * 60)
print("SYNTHETIC DEMO SUMMARY")
print("=" * 60)
print(f"\nData type: {manifest['data_type'].upper()}")
print(f"Generated from: LifeSnaps Fitbit distributions + published norms")
print(f"Users: {manifest['n_users']}, Days: {manifest['n_days']}")
print(f"\nNormalized tables:")
for name, info in manifest["tables"].items():
    print(f"  {name}: {info['rows']} rows")
print(f"\nConflicts demonstrated:")
for c in manifest["conflicts_introduced"]:
    print(f"  - {c}")
print(f"\nThis data is for DEMO PURPOSES ONLY.")
print(f"Real analysis uses the LifeSnaps subset in ../data/subset/")